# Phase 5: Price Analysis
## Deep Dive into Pricing Patterns
**Objective:** Analyze pricing strategies, price ranges, and cost-quality relationships
**Output:** Dashboard-ready pricing insights and segmentation

In [1]:
# Import Required Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)

print("✓ Libraries imported")

✓ Libraries imported


In [2]:
# Load cleaned data
df = pd.read_csv('Zomato_datasets/zomato_cleaned.csv')

# Create price segments
df['price_segment'] = pd.qcut(
    df['avg_cost'],
    q=4,
    labels=['Budget', 'Mid-Range', 'Premium', 'Luxury'],
    duplicates='drop'
)

print(f"Dataset loaded: {df.shape}")
print(f"Price Range: ₹{df['avg_cost'].min():.0f} - ₹{df['avg_cost'].max():.0f}")

Dataset loaded: (7105, 13)
Price Range: ₹40 - ₹6000


In [3]:
# 1. Price Segment Distribution
segment_dist = df['price_segment'].value_counts().sort_index()

fig = px.pie(
    values=segment_dist.values,
    names=segment_dist.index,
    title='Restaurant Distribution by Price Segment',
    color_discrete_sequence=['#3498db', '#2ecc71', '#f39c12', '#e74c3c']
)

fig.update_traces(textposition='inside', textinfo='percent+label')
fig.update_layout(height=600)
fig.show()

print("Price Segment Distribution:")
print(segment_dist)
print(f"\nPercentage:")
print((segment_dist / len(df) * 100).round(2))

Price Segment Distribution:
price_segment
Budget       2708
Mid-Range    1254
Premium      1453
Luxury       1690
Name: count, dtype: int64

Percentage:
price_segment
Budget       38.11
Mid-Range    17.65
Premium      20.45
Luxury       23.79
Name: count, dtype: float64


In [4]:
# 2. Price Distribution by Restaurant Type (Top 10)
top_types = df['restaurant_type'].value_counts().head(10).index
df_top = df[df['restaurant_type'].isin(top_types)]

fig = px.box(
    df_top,
    x='restaurant_type',
    y='avg_cost',
    color='restaurant_type',
    title='Price Distribution by Top 10 Restaurant Types',
    labels={'restaurant_type': 'Restaurant Type', 'avg_cost': 'Average Cost (₹)'},
    points='outliers'
)

fig.update_layout(height=600, xaxis_tickangle=-45, showlegend=False)
fig.show()

print("\nPrice Statistics by Restaurant Type (Top 10):")
price_stats = df_top.groupby('restaurant_type')['avg_cost'].agg(['mean', 'median', 'min', 'max']).round(0)
print(price_stats.sort_values('mean', ascending=False))


Price Statistics by Restaurant Type (Top 10):
                      mean  median    min     max
restaurant_type                                  
Bar                 1278.0  1200.0  400.0  2800.0
Casual Dining, Bar  1134.0  1100.0  400.0  2500.0
Casual Dining        750.0   700.0  200.0  2500.0
Cafe                 588.0   600.0   50.0  2200.0
Delivery             419.0   400.0  100.0  1000.0
Takeaway, Delivery   391.0   400.0  100.0  1600.0
Bakery               349.0   300.0  100.0   900.0
Dessert Parlor       316.0   300.0  100.0  1250.0
Quick Bites          306.0   300.0   40.0  1000.0
Beverage Shop        206.0   200.0  100.0   600.0


In [5]:
# 3. Price vs Rating Relationship
fig = px.scatter(
    df,
    x='avg_cost',
    y='rating',
    color='price_segment',
    size='num_ratings',
    hover_name='restaurant_name',
    title='Price vs Rating Analysis',
    labels={'avg_cost': 'Average Cost (₹)', 'rating': 'Rating', 'price_segment': 'Price Segment'},
    color_discrete_map={'Budget': '#3498db', 'Mid-Range': '#2ecc71', 'Premium': '#f39c12', 'Luxury': '#e74c3c'}
)

fig.update_layout(height=600)
fig.show()

corr = df['avg_cost'].corr(df['rating'])
print(f"\nCorrelation between Price and Rating: {corr:.3f}")


Correlation between Price and Rating: 0.374


In [6]:
# 4. Average Cost by Area (Top 15)
area_price = df.groupby('area').agg({
    'avg_cost': ['mean', 'count'],
    'rating': 'mean'
}).reset_index()

area_price.columns = ['area', 'avg_cost', 'restaurant_count', 'avg_rating']
area_price = area_price[area_price['restaurant_count'] >= 5]
area_price = area_price.sort_values('avg_cost', ascending=False).head(15)

fig = px.bar(
    area_price,
    x='avg_cost',
    y='area',
    color='avg_cost',
    title='Top 15 Most Expensive Areas (Average Cost)',
    labels={'avg_cost': 'Average Cost (₹)', 'area': 'Area'},
    color_continuous_scale='Reds'
)

fig.update_layout(height=600, showlegend=False)
fig.show()

print("\nTop 15 Most Expensive Areas:")
print(area_price[['area', 'avg_cost', 'restaurant_count', 'avg_rating']])


Top 15 Most Expensive Areas:
                     area    avg_cost  restaurant_count  avg_rating
20           Lavelle Road  860.638298               141    3.630496
4            Brigade Road  835.969828               464    3.684267
7           Church Street  745.454545                77    3.606494
29             Whitefield  700.574713               261    3.512261
11            Indiranagar  669.340659               455    3.592308
25       Old Airport Road  612.637363                91    3.409890
22           Malleshwaram  577.661692               402    3.629602
21                MG Road  568.750000                32    3.540625
16  Koramangala 4th Block  566.604938               162    3.516049
3               Bellandur  544.958449               361    3.475900
5             Brookefield  526.834382               477    3.445493
23           Marathahalli  519.807692               260    3.407692
8         Electronic City  512.034739               403    3.363524
15           Kamma

In [7]:
# 5. Price Segment Analysis
segment_analysis = df.groupby('price_segment').agg({
    'avg_cost': ['min', 'max', 'mean'],
    'rating': 'mean',
    'num_ratings': 'mean',
    'has_online_order': 'mean',
    'has_table_booking': 'mean'
}).round(2)

print("\n" + "="*70)
print("PRICE SEGMENT ANALYSIS")
print("="*70)
print(segment_analysis)

# Create comparative visualization
segment_summary = df.groupby('price_segment').agg({
    'rating': 'mean',
    'num_ratings': 'mean'
}).reset_index()

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Avg Rating by Price Segment', 'Avg Num Ratings by Price Segment')
)

fig.add_trace(
    go.Bar(x=segment_summary['price_segment'], y=segment_summary['rating'], name='Rating', marker_color='skyblue'),
    row=1, col=1
)

fig.add_trace(
    go.Bar(x=segment_summary['price_segment'], y=segment_summary['num_ratings'], name='Num Ratings', marker_color='lightcoral'),
    row=1, col=2
)

fig.update_xaxes(title_text='Price Segment', row=1, col=1)
fig.update_xaxes(title_text='Price Segment', row=1, col=2)
fig.update_yaxes(title_text='Average Rating', row=1, col=1)
fig.update_yaxes(title_text='Average Num Ratings', row=1, col=2)

fig.update_layout(height=500, title_text='Price Segment Comparative Analysis', showlegend=False)
fig.show()


PRICE SEGMENT ANALYSIS
              avg_cost                  rating num_ratings has_online_order  \
                   min     max     mean   mean        mean             mean   
price_segment                                                                 
Budget            40.0   300.0   235.19   3.40       55.20             0.49   
Mid-Range        330.0   400.0   389.35   3.45       81.58             0.56   
Premium          450.0   600.0   528.88   3.47      137.09             0.63   
Luxury           650.0  6000.0  1146.24   3.79      527.41             0.46   

              has_table_booking  
                           mean  
price_segment                    
Budget                     0.00  
Mid-Range                  0.00  
Premium                    0.03  
Luxury                     0.41  


C:\Users\Dell\AppData\Local\Temp\ipykernel_8780\1872097834.py:2: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.

C:\Users\Dell\AppData\Local\Temp\ipykernel_8780\1872097834.py:16: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



In [8]:
# 6. Price Distribution (Histogram with KDE)
fig = go.Figure()

fig.add_trace(go.Histogram(
    x=df['avg_cost'],
    nbinsx=50,
    name='Distribution',
    marker_color='lightblue'
))

fig.update_layout(
    title='Overall Price Distribution',
    xaxis_title='Average Cost (₹)',
    yaxis_title='Number of Restaurants',
    height=500,
    showlegend=False
)

fig.show()

print(f"\nPrice Distribution Statistics:")
print(f"  Mean: ₹{df['avg_cost'].mean():.0f}")
print(f"  Median: ₹{df['avg_cost'].median():.0f}")
print(f"  Std Dev: ₹{df['avg_cost'].std():.0f}")
print(f"  Skewness: {stats.skew(df['avg_cost']):.3f}")


Price Distribution Statistics:
  Mean: ₹539
  Median: ₹400
  Std Dev: ₹461
  Skewness: 3.186


In [9]:
# 7. Price Segment by Online Order Availability
price_online = df.groupby(['price_segment', 'has_online_order']).size().unstack(fill_value=0)
price_online.columns = ['No Online Order', 'Online Order Available']

fig = px.bar(
    price_online.reset_index(),
    x='price_segment',
    y=['No Online Order', 'Online Order Available'],
    title='Price Segment vs Online Order Availability',
    labels={'price_segment': 'Price Segment', 'value': 'Count'},
    barmode='group',
    color_discrete_sequence=['#e74c3c', '#2ecc71']
)

fig.update_layout(height=500, xaxis_title='Price Segment')
fig.show()

print("\nPrice Segment vs Online Order Availability:")
print(price_online)

C:\Users\Dell\AppData\Local\Temp\ipykernel_8780\2637882915.py:2: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.




Price Segment vs Online Order Availability:
               No Online Order  Online Order Available
price_segment                                         
Budget                    1381                    1327
Mid-Range                  548                     706
Premium                    536                     917
Luxury                     913                     777


In [1]:
print("\n Price Analysis Complete!")


 Price Analysis Complete!
